# Exoplanet transits: Lomb-Scargle period and phase folding

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk
from astropy.timeseries import LombScargle

## 1. Choose a star and download its latest TESS light curve

In [ ]:
star = "WASP-18"   # <-- change the star name here

search = lk.search_lightcurve(star, mission="TESS", author="SPOC")
print(search)

lc = search[-1].download()          # last entry = latest sector
lc = lc.remove_nans().normalize()
lc.plot();

## 2. Find the period with Lomb-Scargle

In [ ]:
lc_flat = lc.flatten(window_length=901)   # remove slow stellar/instrument trends

t = lc_flat.time.value
f = lc_flat.flux.value

frequency, power = LombScargle(t, f).autopower(minimum_frequency=1/15, maximum_frequency=1/0.3, samples_per_peak=1000)
period = 1 / frequency[np.argmax(power)]

# Lomb-Scargle sometimes picks half the true period for transits.
# If the folded plot below shows two dips, set this to 2.
period = period * 1

plt.plot(1/frequency, power)
plt.xlabel("Period (days)"); plt.ylabel("Power")
plt.show()
print("Period =", period, "days")

## 3. Find T0 and phase-fold

In [ ]:
# T0, step 1: fold and take the deepest binned point as a rough mid-transit
temp = lc_flat.fold(period=period, epoch_time=t[0]).bin(time_bin_size=0.005)
t0 = t[0] + temp.time.value[np.nanargmin(temp.flux.value)]

# T0, step 2: refold at that guess and take the centre of all in-transit bins
temp = lc_flat.fold(period=period, epoch_time=t0).bin(time_bin_size=0.005)
depth = 1 - np.nanmin(temp.flux.value)
in_transit = temp.flux.value < 1 - depth/2
t0 = t0 + np.mean(temp.time.value[in_transit])
print("T0 =", t0, "BTJD")

# Re-flatten with the transits masked so the flattening does not eat the dip
duration = 2 * np.ptp(temp.time.value[in_transit])   # generous transit width
mask = lc.create_transit_mask(period=period, transit_time=t0, duration=duration)
lc_flat = lc.flatten(window_length=901, mask=mask)

folded = lc_flat.fold(period=period, epoch_time=t0)
ax = folded.scatter(s=1, alpha=0.3)
folded.bin(time_bin_size=0.005).scatter(ax=ax, color="red", s=8, label="binned");